# Milvus Embedding Ingestion & Semantic Search

This notebook mirrors `exploitation_zone/milvus_embeddings.py` and walks through
the full pipeline interactively:

1. **Setup** — connect to MinIO and Milvus.
2. **Collections** — create the three vector collections.
3. **Audio embeddings** — PANNs CNN14 (2048-dim) for similarity search.
4. **Text embeddings** — all-MiniLM-L6-v2 (384-dim) for RAG chatbot.
5. **Cymatics image embeddings** — CLIP ViT-B/32 (512-dim) for visual pattern search.
6. **Full ingestion** — paginated, idempotent upsert.
7. **Similarity search** — find acoustically similar sounds.
8. **Text search** — semantic search for the RAG chatbot.
9. **Cymatics pattern search** — find visually similar patterns + text-to-image.

**Architecture**: Milvus sits in the Exploitation Zone as a specialised vector store
alongside MinIO Delta Lake, following the project's polyglot-persistence design
(option 2 from the project statement).

| Collection | Model | Dim | Use Case |
|---|---|---|---|
| `sound_audio_embeddings` | PANNs CNN14 | 2048 | Audio similarity search & classification |
| `sound_text_embeddings` | all-MiniLM-L6-v2 | 384 | RAG chatbot (questions about features) |
| `sound_cymatics_embeddings` | CLIP ViT-B/32 | 512 | Visual pattern search (image & text queries) |

## 1. Setup & Connections

In [1]:
import os, sys

# Project root so we can import shared helpers and the milvus_embeddings module.
_NB_DIR  = os.path.abspath("")
_EZ_DIR  = os.path.abspath(os.path.join(_NB_DIR, ".."))
_PROJECT = os.path.abspath(os.path.join(_EZ_DIR, ".."))
for p in (_PROJECT, _EZ_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(os.path.join(_PROJECT, ".env"))

import numpy as np
import csv, io, json

In [2]:
# ---- MinIO ----
from shared.minio_helpers import create_minio_client

minio_client = create_minio_client()
print("MinIO connected:", minio_client.list_buckets())

MinIO connected: [Bucket(name='exploitation-zone', creation_date=datetime.datetime(2026, 5, 19, 3, 16, 18, 96000, tzinfo=datetime.timezone.utc)), Bucket(name='landing-zone', creation_date=datetime.datetime(2026, 5, 18, 15, 41, 31, 793000, tzinfo=datetime.timezone.utc)), Bucket(name='trusted-zone', creation_date=datetime.datetime(2026, 5, 18, 15, 40, 17, 102000, tzinfo=datetime.timezone.utc))]


In [3]:
# ---- Milvus ----
from pymilvus import MilvusClient

MILVUS_HOST = os.environ.get("MILVUS_HOST", "localhost")
MILVUS_PORT = os.environ.get("MILVUS_PORT", "19530")
MILVUS_URI  = f"http://{MILVUS_HOST}:{MILVUS_PORT}"

milvus_client = MilvusClient(uri=MILVUS_URI)
print(f"Milvus connected at {MILVUS_URI}")
print("Existing collections:", milvus_client.list_collections())

Milvus connected at http://localhost:19530
Existing collections: ['sound_audio_embeddings', 'sound_text_embeddings', 'sound_cymatics_embeddings']


## 2. Create Milvus Collections

We define three collections, each with an **HNSW** index using **COSINE** distance:

- `sound_audio_embeddings` — 2048-dim PANNs CNN14 vectors + scalar metadata for filtering.
- `sound_text_embeddings` — 384-dim sentence-transformer vectors + the generated description text.
- `sound_cymatics_embeddings` — 512-dim CLIP ViT-B/32 vectors from cymatics images + image path.

In [4]:
from milvus_embeddings import (
    AUDIO_COLLECTION,
    TEXT_COLLECTION,
    CYMATICS_COLLECTION,
    AUDIO_EMBEDDING_DIM,
    TEXT_EMBEDDING_DIM,
    CYMATICS_EMBEDDING_DIM,
    create_audio_collection,
    create_text_collection,
    create_cymatics_collection,
)

create_audio_collection(milvus_client)
create_text_collection(milvus_client)
create_cymatics_collection(milvus_client)

print("\nCollections:", milvus_client.list_collections())

  Collection 'sound_audio_embeddings' exists (38 entities) — skipping creation.
  Collection 'sound_text_embeddings' exists (38 entities) — skipping creation.
  Collection 'sound_cymatics_embeddings' exists (0 entities) — skipping creation.

Collections: ['sound_audio_embeddings', 'sound_text_embeddings', 'sound_cymatics_embeddings']


You can also verify the collections in the **Attu** web interface at http://localhost:3000.

Navigate to **Explorer** and check the schema, index configuration, and current entity count.

## 3. Load Exploitation-Zone Metadata

In [5]:
from milvus_embeddings import load_exploitation_metadata, EXPLOITATION_BUCKET, METADATA_KEY

rows = load_exploitation_metadata(minio_client)
print(f"\nLoaded {len(rows)} rows. Columns: {list(rows[0].keys()) if rows else '(empty)'}")

  Loaded 32 rows from exploitation-zone/metadata/observations.csv

Loaded 32 rows. Columns: ['uuid', 'source_id', 'time_recorded/added', 'processing_version', 'feature_version', 'duration', 'symmetry_score', 'pattern_stability_score', 'image_resolution', 'video_resolution', 'audio_size', 'image_size', 'video_size', 'processing_time(seconds)', 'device', 'audio_format', 'source', 'peak_frequency_hz', 'peak_time_s', 'peak_amplitude', 'peak_rms', 'audio_path', 'image_path', 'video_path', 'all_peak_frequencies_hz', 'tags', 'description', 'category', 'spectral_centroid_hz', 'spectral_bandwidth_hz', 'spectral_rolloff_hz', 'spectral_flatness', 'signal_energy', 'spectral_entropy', 'zero_crossing_rate', 'loudness', 'MFCCs', 'harmonic_energy_ratio', 'feature_processing_time(seconds)']


In [6]:
# Quick look at the first record.
if rows:
    for k, v in rows[0].items():
        print(f"  {k:.<40s} {str(v)[:80]}")

  uuid.................................... 00a11728-db7d-44b1-912f-1d2e4b0899c8
  source_id............................... 485373
  time_recorded/added..................... 2019-09-18T16:59:28Z
  processing_version...................... 2.0.0
  feature_version......................... 1.0.0
  duration................................ 8.49047619047619
  symmetry_score.......................... 0.9594778915503759
  pattern_stability_score................. 0.982891766782357
  image_resolution........................ 2048x2048
  video_resolution........................ 600x600
  audio_size.............................. 748904
  image_size.............................. 2547166
  video_size.............................. 1318977
  processing_time(seconds)................ 6.667760417010868
  device.................................. -
  audio_format............................ wav
  source.................................. Freesound
  peak_frequency_hz....................... 56.0
  peak_time_s..

## 4. Audio Embeddings — PANNs CNN14 (2048-dim)

**PANNs CNN14** (Pretrained Audio Neural Networks) is the standard model for
environmental sound classification. It was trained on AudioSet and produces
2048-dimensional embeddings that capture acoustic characteristics.

Our audio is at 44.1 kHz; PANNs expects 32 kHz, so we resample with `scipy.signal.resample`.

### 4.1 Single-record demo

In [7]:
from milvus_embeddings import (
    compute_audio_embedding,
    _load_audio_float32,
    TRUSTED_BUCKET,
)

# Pick the first row with an audio path.
demo_row = next((r for r in rows if (r.get("audio_path") or "").strip()), None)
if demo_row:
    audio_path = demo_row["audio_path"].strip()
    print(f"Demo record: uuid={demo_row['uuid'][:8]}, audio_path={audio_path}")

    resp = minio_client.get_object(TRUSTED_BUCKET, audio_path)
    wav_bytes = resp.read(); resp.close(); resp.release_conn()
    sr, audio = _load_audio_float32(wav_bytes)
    print(f"Audio: {len(audio)} samples, {sr} Hz, {len(audio)/sr:.1f}s")

    embedding = compute_audio_embedding(audio, sr)
    print(f"Embedding shape: ({len(embedding)},)")
    print(f"First 10 values: {embedding[:10]}")
else:
    print("No row with audio_path found.")

Demo record: uuid=00a11728, audio_path=audio/56/00a11728-db7d-44b1-912f-1d2e4b0899c8-56.wav
Audio: 374430 samples, 44100 Hz, 8.5s
  Loading PANNs CNN14 model...
Checkpoint path: /Users/arman/panns_data/Cnn14_mAP=0.431.pth
Using CPU.
  PANNs CNN14 ready.
Embedding shape: (2048,)
First 10 values: [0.0, 0.3178820013999939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


### 4.2 Batch audio ingestion (idempotent upsert)

We use `upsert` (not `insert`) so that running this cell multiple times on the same
rows leaves Milvus in exactly the same state — matching the lab’s idempotent
`ingest_batch` pattern.

In [8]:
from milvus_embeddings import ingest_audio_batch

# Start with a small page to verify.
page = rows[:3]
audio_count = ingest_audio_batch(milvus_client, minio_client, page)
print(f"\nAudio embeddings upserted: {audio_count}")

    [1/3] uuid=00a11728 — audio OK (0.1s)
    [2/3] uuid=09bfde40 — audio OK (0.1s)
    [3/3] uuid=13159abb — audio OK (0.1s)
  Upserted 3 audio embeddings into 'sound_audio_embeddings'

Audio embeddings upserted: 3


In [9]:
# Idempotency check: run the same page again.
audio_count_2 = ingest_audio_batch(milvus_client, minio_client, page)
stats = milvus_client.get_collection_stats(AUDIO_COLLECTION)
print(f"\nAfter second upsert: {stats.get('row_count', '?')} entities (should be same as first run)")

    [1/3] uuid=00a11728 — audio OK (0.1s)
    [2/3] uuid=09bfde40 — audio OK (0.1s)
    [3/3] uuid=13159abb — audio OK (0.1s)
  Upserted 3 audio embeddings into 'sound_audio_embeddings'

After second upsert: 44 entities (should be same as first run)


## 5. Text Embeddings — all-MiniLM-L6-v2 (384-dim)

For the RAG chatbot we build natural-language descriptions from metadata fields
(category, tags, peak frequency, spectral features, cymatics scores) and embed
them with the same sentence-transformer model used in the lab.

This lets users ask questions like *"what frequency do sea waves usually have?"*
or *"which sounds have the most complex patterns?"*

### 5.1 Description construction demo

In [10]:
from milvus_embeddings import build_sound_description

if rows:
    desc = build_sound_description(rows[0])
    print(f"UUID: {rows[0].get('uuid', '?')[:8]}")
    print(f"\nGenerated description ({len(desc)} chars):\n")
    print(desc)

UUID: 00a11728

Generated description (649 chars):

This is a cat purring sound. Recorded via Freesound. Sound of a male cat purring while laying on a bed. Recorded on a Tascam DR-100MKIII with an Audio Technica AT875R shotgun mic in 24/48 mono. There was a slight modification of amplitude using Adobe Audition. Tags: cat|domesticated|field-recording|happy|kitty|pet|purr|purring. Peak frequency: 56.0 Hz. This is a low-frequency sound. Spectral centroid: 59.5 Hz. The sound has high spectral complexity (noise-like). Cymatics symmetry score: 0.959. Produces highly symmetric cymatics patterns. Pattern stability: 0.983. Harmonic energy ratio: 0.686. The sound is highly harmonic. Loudness: -10.7 dB.


In [11]:
from milvus_embeddings import compute_text_embeddings

sample_texts = [build_sound_description(r) for r in rows[:3]]
embeddings = compute_text_embeddings(sample_texts)

print(f"Batch of {len(embeddings)} texts embedded.")
print(f"Each vector: {len(embeddings[0])}-dim")
print(f"First vector (first 10): {embeddings[0][:10]}")

  Loading text model 'all-MiniLM-L6-v2'...


I0530 17:03:30.863343 7462445 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(82, generation: 1)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Text model ready.
Batch of 3 texts embedded.
Each vector: 384-dim
First vector (first 10): [0.038044508546590805, -0.04729462042450905, -0.029368042945861816, -0.061583127826452255, -0.15356120467185974, 0.009863808751106262, 0.033090781420469284, -0.06857123225927353, -0.06369368731975555, -0.0003602945653256029]


### 5.2 Batch text ingestion

In [12]:
from milvus_embeddings import ingest_text_batch

text_count = ingest_text_batch(milvus_client, page)
print(f"\nText embeddings upserted: {text_count}")

  Embedding 3 descriptions in one vectorised batch...
  Upserted 3 text embeddings into 'sound_text_embeddings'

Text embeddings upserted: 3


## 6. Cymatics Image Embeddings — CLIP ViT-B/32 (512-dim)

**CLIP** (Contrastive Language-Image Pretraining) maps images and text into the
same 512-dimensional embedding space. This enables two powerful search modes:

- **Image → Image**: upload a cymatics image and find visually similar patterns.
- **Text → Image**: ask "star shaped pattern" or "which pattern does a bird sound have?"
  and retrieve matching cymatics images directly.

We embed the 2048×2048 cymatics PNG from the trusted zone for each record.

### 6.1 Single-record demo

In [13]:
from milvus_embeddings import (
    compute_cymatics_embedding,
    compute_cymatics_text_embedding,
    TRUSTED_BUCKET,
)

# Pick the first row with an image path.
demo_row = next((r for r in rows if (r.get("image_path") or "").strip()), None)
if demo_row:
    image_path = demo_row["image_path"].strip()
    print(f"Demo record: uuid={demo_row['uuid'][:8]}, image_path={image_path}")

    resp = minio_client.get_object(TRUSTED_BUCKET, image_path)
    image_bytes = resp.read(); resp.close(); resp.release_conn()
    print(f"Image size: {len(image_bytes) / 1024:.1f} KB")

    embedding = compute_cymatics_embedding(image_bytes)
    print(f"Embedding shape: ({len(embedding)},)")
    print(f"First 10 values: {embedding[:10]}")
else:
    print("No row with image_path found.")

Demo record: uuid=00a11728, image_path=images/56/00a11728-db7d-44b1-912f-1d2e4b0899c8-56.png
Image size: 2487.5 KB
  Loading CLIP model 'openai/clip-vit-base-patch32'...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  CLIP model ready.
Embedding shape: (512,)
First 10 values: [-0.02789813093841076, 0.004182880744338036, -0.0025514850858598948, 0.008943680673837662, -0.004980565048754215, -0.040037233382463455, -0.011632461100816727, 0.01587408222258091, 0.035901423543691635, 0.001523311948403716]


In [14]:
# CLIP text embedding — same 512-dim space as image embeddings.
text_emb = compute_cymatics_text_embedding("circular symmetric pattern")
print(f"Text embedding shape: ({len(text_emb)},)")
print(f"First 10 values: {text_emb[:10]}")

Text embedding shape: (512,)
First 10 values: [-0.02086884342133999, 0.018721725791692734, 0.01283130794763565, 0.03220628947019577, -0.005836357828229666, -0.009003724902868271, -0.012299670837819576, -0.09877543151378632, 0.009843207895755768, -0.036422666162252426]


### 6.2 Batch cymatics ingestion (idempotent upsert)

In [15]:
from milvus_embeddings import ingest_cymatics_batch

# Start with a small page to verify.
page = rows[:3]
cymatics_count = ingest_cymatics_batch(milvus_client, minio_client, page)
print(f"\nCymatics embeddings upserted: {cymatics_count}")

    [1/3] uuid=00a11728 — cymatics OK (0.2s)
    [2/3] uuid=09bfde40 — cymatics OK (0.1s)
    [3/3] uuid=13159abb — cymatics OK (0.1s)
  Upserted 3 cymatics embeddings into 'sound_cymatics_embeddings'

Cymatics embeddings upserted: 3


## 7. Full Ingestion

Process all exploitation-zone records across all three collections. For larger
datasets you can paginate (the `ingest_*_batch` functions are idempotent, so
re-running is safe).

In [16]:
# Drop and recreate clean collections
milvus_client.drop_collection(AUDIO_COLLECTION)
milvus_client.drop_collection(TEXT_COLLECTION)
milvus_client.drop_collection(CYMATICS_COLLECTION)
create_audio_collection(milvus_client)
create_text_collection(milvus_client)
create_cymatics_collection(milvus_client)

PAGE_SIZE = 50

total_audio    = 0
total_text     = 0
total_cymatics = 0

for start in range(0, len(rows), PAGE_SIZE):
    page = rows[start : start + PAGE_SIZE]
    print(f"\n--- Page {start // PAGE_SIZE + 1} ({len(page)} rows) ---")
    total_audio    += ingest_audio_batch(milvus_client, minio_client, page)
    total_text     += ingest_text_batch(milvus_client, page)
    total_cymatics += ingest_cymatics_batch(milvus_client, minio_client, page)


milvus_client.flush(AUDIO_COLLECTION)
milvus_client.flush(TEXT_COLLECTION)
milvus_client.flush(CYMATICS_COLLECTION)

for col in (AUDIO_COLLECTION, TEXT_COLLECTION, CYMATICS_COLLECTION):
    stats = milvus_client.get_collection_stats(col)
    print(f"  {col}: {stats.get('row_count', '?')} entities")

print(f"\n=== Ingestion complete ===")
print(f"  Audio embeddings:    {total_audio}")
print(f"  Text embeddings:     {total_text}")
print(f"  Cymatics embeddings: {total_cymatics}")

  Created collection 'sound_audio_embeddings' (dim=2048, HNSW/COSINE)
  Created collection 'sound_text_embeddings' (dim=384, HNSW/COSINE)
  Created collection 'sound_cymatics_embeddings' (dim=512, HNSW/COSINE)

--- Page 1 (32 rows) ---
    [1/32] uuid=00a11728 — audio OK (0.3s)
    [2/32] uuid=09bfde40 — audio OK (0.1s)
    [3/32] uuid=13159abb — audio OK (0.1s)
    [4/32] uuid=31dac2a1 — audio OK (0.1s)
    [5/32] uuid=3bbf9f45 — audio OK (0.0s)
    [6/32] uuid=3d6ab8ab — audio OK (0.1s)
    [7/32] uuid=56a22272 — audio OK (0.1s)
    [8/32] uuid=5706b0f9 — audio OK (0.0s)
    [9/32] uuid=60d4baa2 — audio OK (0.0s)
    [10/32] uuid=6eda6370 — audio OK (0.1s)
    [11/32] uuid=75600263 — audio OK (0.1s)
    [12/32] uuid=789ced0f — audio OK (0.1s)
    [13/32] uuid=a2f5f95b — audio OK (0.1s)
    [14/32] uuid=a6899d2e — audio OK (0.1s)
    [15/32] uuid=b5bcd45c — audio OK (0.1s)
    [16/32] uuid=b637aadd — audio OK (0.0s)
    [17/32] uuid=c45ba243 — audio OK (0.1s)
    [18/32] uuid=c7f54155

In [17]:
# Verify entity counts in Milvus.
for col in (AUDIO_COLLECTION, TEXT_COLLECTION, CYMATICS_COLLECTION):
    stats = milvus_client.get_collection_stats(col)
    print(f"  {col}: {stats.get('row_count', '?')} entities")

  sound_audio_embeddings: 32 entities
  sound_text_embeddings: 32 entities
  sound_cymatics_embeddings: 32 entities


## 8. Audio Similarity Search

Given a query sound, find the most acoustically similar recordings using
approximate nearest-neighbour search on the PANNs CNN14 embeddings.

In [18]:
from milvus_embeddings import search_similar_sounds

# Use the first record as the query.
if rows:
    query_row = rows[0]
    audio_path = query_row["audio_path"].strip()
    resp = minio_client.get_object(TRUSTED_BUCKET, audio_path)
    wav_bytes = resp.read(); resp.close(); resp.release_conn()
    sr, query_audio = _load_audio_float32(wav_bytes)

    print(f"Query: uuid={query_row['uuid'][:8]}, category={query_row.get('category','?')}")
    print(f"Finding 5 most similar sounds...\n")

    results = search_similar_sounds(milvus_client, query_audio, sr, top_k=5)
    for i, hit in enumerate(results):
        print(
            f"  {i+1}. uuid={hit['entity']['uuid'][:8]}  "
            f"category={hit['entity'].get('category','?'):.<20s}  "
            f"distance={hit['distance']:.4f}  "
            f"peak_freq={hit['entity'].get('peak_frequency_hz', 0):.0f} Hz"
        )

Query: uuid=00a11728, category=cat purring
Finding 5 most similar sounds...

  1. uuid=00a11728  category=cat purring.........  distance=1.0000  peak_freq=56 Hz
  2. uuid=f9ca7ee1  category=cat purring.........  distance=0.6107  peak_freq=8 Hz
  3. uuid=2f2d8657  category=dog bark............  distance=0.5874  peak_freq=528 Hz
  4. uuid=8152b77a  category=dog bark............  distance=0.5540  peak_freq=504 Hz
  5. uuid=e28f9248  category=dog bark............  distance=0.5404  peak_freq=308 Hz


### 8.1 Filtered similarity search

Milvus supports scalar pre-filters on the ANN search. For example, restrict
results to a specific frequency range.

In [19]:
if rows:
    results_filtered = search_similar_sounds(
        milvus_client, query_audio, sr, top_k=5,
        filter_expr="peak_frequency_hz > 200 and peak_frequency_hz < 2000",
    )
    print("Similar sounds (200–2000 Hz only):\n")
    for i, hit in enumerate(results_filtered):
        print(
            f"  {i+1}. uuid={hit['entity']['uuid'][:8]}  "
            f"category={hit['entity'].get('category','?'):.<20s}  "
            f"distance={hit['distance']:.4f}  "
            f"peak_freq={hit['entity'].get('peak_frequency_hz', 0):.0f} Hz"
        )

Similar sounds (200–2000 Hz only):

  1. uuid=2f2d8657  category=dog bark............  distance=0.5874  peak_freq=528 Hz
  2. uuid=8152b77a  category=dog bark............  distance=0.5540  peak_freq=504 Hz
  3. uuid=e28f9248  category=dog bark............  distance=0.5404  peak_freq=308 Hz
  4. uuid=b9e86afb  category=dog bark............  distance=0.5277  peak_freq=604 Hz
  5. uuid=3d6ab8ab  category=singing bowl........  distance=0.4923  peak_freq=540 Hz


## 9. Text Semantic Search (RAG Chatbot Retrieval)

Search the text-embedding collection with natural-language queries.
These results power the RAG chatbot — the retrieved descriptions provide
context for an LLM to answer user questions about sound characteristics.

In [32]:
from milvus_embeddings import search_by_text

queries = [
    "what frequency do cat purring usually have?",
    "which sounds have the most complex patterns?",
    "find sounds with highly symmetric cymatics",
    "low frequency harmonic sound",
]

for q in queries:
    print(f"\nQuery: \"{q}\"")
    print("-" * 70)
    results = search_by_text(milvus_client, q, top_k=3)
    for i, hit in enumerate(results):
        desc = hit["entity"].get("description_text", "")[:120]
        print(
            f"  {i+1}. [{hit['distance']:.4f}] "
            f"category={hit['entity'].get('category','?')}  "
            f"freq={hit['entity'].get('peak_frequency_hz', 0):.0f} Hz"
        )
        print(f"     {desc}...")


Query: "what frequency do cat purring usually have?"
----------------------------------------------------------------------
  1. [0.6590] category=cat purring  freq=56 Hz
     This is a cat purring sound. Recorded via Freesound. Sound of a male cat purring while laying on a bed. Recorded on a Ta...
  2. [0.6472] category=cat purring  freq=116 Hz
     This is a cat purring sound. Recorded via Freesound. Recorded up close what my cat's purring sounds like. Edited to only...
  3. [0.6381] category=cat purring  freq=8 Hz
     This is a cat purring sound. Recorded via Freesound. Male cat purring variation 2 Tags: OWI|cat|field-recording|male|pur...

Query: "which sounds have the most complex patterns?"
----------------------------------------------------------------------
  1. [0.4886] category=  freq=48 Hz
     Recorded via hot-path. Peak frequency: 48.0 Hz. This is a low-frequency sound. Spectral centroid: 268.4 Hz. The sound ha...
  2. [0.4856] category=  freq=56 Hz
     Recorded via ho

### 9.1 Filtered text search

Combine semantic search with scalar filters.

In [ ]:
results_filtered = search_by_text(
    milvus_client,
    "bright sounding high energy",
    top_k=5,
    filter_expr='source == "Freesound"',
)

print('Query: "bright sounding high energy" (Freesound only)\n')
for i, hit in enumerate(results_filtered):
    print(
        f"  {i+1}. [{hit['distance']:.4f}] "
        f"category={hit['entity'].get('category','?')}  "
        f"source={hit['entity'].get('source','?')}"
    )

## 10. Cymatics Pattern Search

CLIP enables two search modes on the cymatics image embeddings:

- **Image-to-image**: given a cymatics image (from MinIO or user upload), find
  recordings whose cymatics patterns look most similar.
- **Text-to-image**: describe a pattern in natural language (e.g. "star shaped
  pattern", "circular rings") and retrieve matching cymatics images.

### 10.1 Image-to-image: find visually similar patterns

In [20]:
from milvus_embeddings import search_similar_patterns

# Use the first record's cymatics image as the query.
if rows:
    query_row = rows[0]
    image_path = query_row["image_path"].strip()
    resp = minio_client.get_object(TRUSTED_BUCKET, image_path)
    query_image = resp.read(); resp.close(); resp.release_conn()

    print(f"Query: uuid={query_row['uuid'][:8]}, category={query_row.get('category','?')}")
    print(f"Image: {image_path} ({len(query_image)/1024:.1f} KB)")
    print(f"Finding 5 most visually similar cymatics patterns...\n")

    results = search_similar_patterns(milvus_client, query_image, top_k=5)
    for i, hit in enumerate(results):
        print(
            f"  {i+1}. uuid={hit['entity']['uuid'][:8]}  "
            f"category={hit['entity'].get('category','?'):.<20s}  "
            f"distance={hit['distance']:.4f}  "
            f"peak_freq={hit['entity'].get('peak_frequency_hz', 0):.0f} Hz"
        )

Query: uuid=00a11728, category=cat purring
Image: images/56/00a11728-db7d-44b1-912f-1d2e4b0899c8-56.png (2487.5 KB)
Finding 5 most visually similar cymatics patterns...

  1. uuid=00a11728  category=cat purring.........  distance=1.0000  peak_freq=56 Hz
  2. uuid=20bb5e5c  category=ocean...............  distance=0.9821  peak_freq=8 Hz
  3. uuid=6eda6370  category=tuning fork.........  distance=0.9811  peak_freq=96 Hz
  4. uuid=f45346b4  category=....................  distance=0.9803  peak_freq=48 Hz
  5. uuid=31dac2a1  category=rain................  distance=0.9799  peak_freq=516 Hz


### 10.2 Text-to-image: describe a pattern in natural language

Because CLIP shares the embedding space between images and text, we can search
cymatics images using natural-language descriptions. This powers queries like
*"which pattern does a bird sound have?"* or *"find star shaped patterns"*.

In [21]:
from milvus_embeddings import search_patterns_by_text

queries = [
    "star shaped pattern",
    "circular symmetric rings",
    "complex chaotic pattern",
    "simple geometric shape",
]

for q in queries:
    print(f"\nQuery: \"{q}\"")
    print("-" * 70)
    results = search_patterns_by_text(milvus_client, q, top_k=3)
    for i, hit in enumerate(results):
        print(
            f"  {i+1}. [{hit['distance']:.4f}] "
            f"category={hit['entity'].get('category','?')}  "
            f"freq={hit['entity'].get('peak_frequency_hz', 0):.0f} Hz  "
            f"symmetry={hit['entity'].get('symmetry_score', 0):.3f}"
        )


Query: "star shaped pattern"
----------------------------------------------------------------------
  1. [0.2628] category=tuning fork  freq=92 Hz  symmetry=0.954
  2. [0.2620] category=waterfall  freq=44 Hz  symmetry=0.966
  3. [0.2603] category=church bells  freq=540 Hz  symmetry=0.964

Query: "circular symmetric rings"
----------------------------------------------------------------------
  1. [0.2947] category=tuning fork  freq=92 Hz  symmetry=0.954
  2. [0.2936] category=church bells  freq=540 Hz  symmetry=0.964
  3. [0.2931] category=crickets  freq=116 Hz  symmetry=0.957

Query: "complex chaotic pattern"
----------------------------------------------------------------------
  1. [0.2724] category=tuning fork  freq=440 Hz  symmetry=0.966
  2. [0.2708] category=church bells  freq=540 Hz  symmetry=0.964
  3. [0.2699] category=bird song  freq=3572 Hz  symmetry=0.959

Query: "simple geometric shape"
----------------------------------------------------------------------
  1. [0.2482] 

### 10.3 Filtered pattern search

Combine visual similarity with scalar filters (e.g. only high-symmetry patterns).

In [ ]:
results_filtered = search_patterns_by_text(
    milvus_client,
    "circular wave pattern",
    top_k=5,
    filter_expr="symmetry_score > 0.8",
)

print('Query: "circular wave pattern" (symmetry > 0.8 only)\n')
for i, hit in enumerate(results_filtered):
    print(
        f"  {i+1}. [{hit['distance']:.4f}] "
        f"category={hit['entity'].get('category','?')}  "
        f"symmetry={hit['entity'].get('symmetry_score', 0):.3f}  "
        f"freq={hit['entity'].get('peak_frequency_hz', 0):.0f} Hz"
    )

## Summary

We have:
1. Created three Milvus collections with HNSW/COSINE indexes.
2. Ingested **PANNs CNN14** audio embeddings (2048-dim) for every exploitation-zone record.
3. Ingested **all-MiniLM-L6-v2** text embeddings (384-dim) from constructed metadata descriptions.
4. Ingested **CLIP ViT-B/32** cymatics image embeddings (512-dim) from trusted-zone cymatics PNGs.
5. Demonstrated **audio similarity search** (with optional scalar filters).
6. Demonstrated **text semantic search** for RAG chatbot retrieval.
7. Demonstrated **cymatics pattern search** — both image-to-image and text-to-image.

All ingestion uses idempotent `upsert` — re-running is safe and converges to the correct state.

**Explore further**: Open the Attu web interface at http://localhost:3000 to browse
collection schemas, inspect entities, and run ad-hoc searches.